In [3]:
import pandas as pd

## <데이터 파일 기본 사항 확인>

- labeled -> train 데이터   
- unlabeled -> test 데이터

### 컬럼 의미 분석

| 구분         | 컬럼                         | 의미                     |
| ---------- | -------------------------- | ---------------------- |
| **Target** | `PassOrFail`               | 제품 품질 합격/불합격 판정        |
| **시간**     | `Injection_Time`           | 사출 소요 시간               |
|            | `Filling_Time`             | 금형 충전 시간               |
|            | `Plasticizing_Time`        | 가소화 소요 시간              |
|            | `Cycle_Time`               | 전체 생산 사이클 시간           |
|            | `Clamp_Close_Time`         | 금형 폐쇄 시간               |
| **위치**     | `Cushion_Position`         | 사출 후 잔류 수지와 관련된 스크류 위치 |
|            | `Plasticizing_Position`    | 가소화 완료 시 스크류 위치        |
|            | `Clamp_Open_Position`      | 금형 개방 위치               |
| **속도**     | `Max_Injection_Speed`      | 최대 사출 속도               |
|            | `Max_Screw_RPM`            | 최대 스크류 회전속도            |
|            | `Average_Screw_RPM`        | 평균 스크류 회전속도            |
| **압력**     | `Max_Injection_Pressure`   | 최대 사출압력                |
|            | `Max_Switch_Over_Pressure` | 사출→보압 절환 시 최대 압력       |
|            | `Max_Back_Pressure`        | 최대 배압                  |
|            | `Average_Back_Pressure`    | 평균 배압                  |
| **온도**     | `Barrel_Temperature_1~6`   | 배럴 각 구간의 온도            |
|            | `Hopper_Temperature`       | 호퍼 온도                  |
|            | `Mold_Temperature_3~4`     | 금형 측정 지점 온도            |


### 1. cn7 파일 기본 사항 확인
 - CN7 = 현대 아반떼 7세대

In [4]:
# cn7 데이터 확인

train_cn7 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv")

In [5]:
# 데이터 크기 확인

train_cn7.shape

(1211, 26)

In [6]:
# 데이터 정보 확인

train_cn7.info()

<class 'pandas.DataFrame'>
RangeIndex: 1211 entries, 0 to 1210
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                1211 non-null   int64  
 1   PassOrFail                1211 non-null   int64  
 2   Injection_Time            1211 non-null   float64
 3   Filling_Time              1211 non-null   float64
 4   Plasticizing_Time         1211 non-null   float64
 5   Cycle_Time                1211 non-null   float64
 6   Clamp_Close_Time          1211 non-null   float64
 7   Cushion_Position          1211 non-null   float64
 8   Plasticizing_Position     1211 non-null   float64
 9   Clamp_Open_Position       1211 non-null   float64
 10  Max_Injection_Speed       1211 non-null   float64
 11  Max_Screw_RPM             1211 non-null   float64
 12  Average_Screw_RPM         1211 non-null   float64
 13  Max_Injection_Pressure    1211 non-null   float64
 14  Max_Switch_Over_Pre

In [ ]:
# 데이터 타입 확인
import numpy as np

num_cn7 = train_cn7.select_dtypes(include = np.number)
num_cn7.columns

Index(['Unnamed: 0', 'PassOrFail', 'Injection_Time', 'Filling_Time',
       'Plasticizing_Time', 'Cycle_Time', 'Clamp_Close_Time',
       'Cushion_Position', 'Plasticizing_Position', 'Clamp_Open_Position',
       'Max_Injection_Speed', 'Max_Screw_RPM', 'Average_Screw_RPM',
       'Max_Injection_Pressure', 'Max_Switch_Over_Pressure',
       'Max_Back_Pressure', 'Average_Back_Pressure', 'Barrel_Temperature_1',
       'Barrel_Temperature_2', 'Barrel_Temperature_3', 'Barrel_Temperature_4',
       'Barrel_Temperature_5', 'Barrel_Temperature_6', 'Hopper_Temperature',
       'Mold_Temperature_3', 'Mold_Temperature_4'],
      dtype='str')

In [11]:
# 결측치 확인

train_cn7.isnull().sum() / len(train_cn7)

Unnamed: 0                  0.0
PassOrFail                  0.0
Injection_Time              0.0
Filling_Time                0.0
Plasticizing_Time           0.0
Cycle_Time                  0.0
Clamp_Close_Time            0.0
Cushion_Position            0.0
Plasticizing_Position       0.0
Clamp_Open_Position         0.0
Max_Injection_Speed         0.0
Max_Screw_RPM               0.0
Average_Screw_RPM           0.0
Max_Injection_Pressure      0.0
Max_Switch_Over_Pressure    0.0
Max_Back_Pressure           0.0
Average_Back_Pressure       0.0
Barrel_Temperature_1        0.0
Barrel_Temperature_2        0.0
Barrel_Temperature_3        0.0
Barrel_Temperature_4        0.0
Barrel_Temperature_5        0.0
Barrel_Temperature_6        0.0
Hopper_Temperature          0.0
Mold_Temperature_3          0.0
Mold_Temperature_4          0.0
dtype: float64

In [30]:
# Target 제외 중복 -> 동일한 공정조건으로 여러 제품이 생산됐는지 확인

train_cn7.drop(columns="PassOrFail").duplicated().sum()

np.int64(0)

In [31]:
# 동일 X + 동일 Target -> 반복 생산 데이터일 가능성
train_cn7.duplicated().sum()

np.int64(0)

In [32]:
# 동일 X + 다른 Target -> 같은 공정조건인데 품질 결과가 달라지는 노이즈 또는 미관측 변수 가능성

train_cn7.groupby(
    list(train_cn7.columns.drop("PassOrFail"))
)["PassOrFail"].nunique().gt(1).sum()

np.int64(0)

In [19]:
# target 분포 확인

train_cn7['PassOrFail'].value_counts()

PassOrFail
0    1194
1      17
Name: count, dtype: int64

In [34]:
# target 분포 비율 확인
train_cn7["PassOrFail"].value_counts(normalize=True)

PassOrFail
0    0.985962
1    0.014038
Name: proportion, dtype: float64

In [39]:
# 시계열 데이터
# 시계열 컬럼 존재 여부 확인
[col for col in train_cn7.columns 
 if any(x in col.lower() for x in ["date", "time", "timestamp"])]

['Injection_Time',
 'Filling_Time',
 'Plasticizing_Time',
 'Cycle_Time',
 'Clamp_Close_Time']

In [ ]:
# 날짜/시각 컬럼 확인 -> 없음을 확인
[col for col in train_cn7.columns 
 if any(x in col.lower() for x in ["date", "datetime", "timestamp"])]

[]

In [43]:
# 설비 구조
# 설비 식별자 컬럼 확인
[col for col in train_cn7.columns
 if any(x in col.lower() for x in ["machine_id", "equipment_id", "device_id", "line_id"])]

[]

In [45]:
# LOT 구조
[col for col in train_cn7.columns
 if any(x in col.lower() for x in ["lot", "batch", "group", "product_id"])]

[]

### 2. rg3 파일 기본 사항 확인
- RG3 = 제네시스 G80 3세대

In [8]:
# rg3 데이터 확인

train_rg3 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_rg3.csv")

In [9]:
# 데이터 크기 확인

train_rg3.shape

(1182, 26)

In [10]:
# 데이터 정보 확인

train_rg3.info()

<class 'pandas.DataFrame'>
RangeIndex: 1182 entries, 0 to 1181
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                1182 non-null   int64  
 1   PassOrFail                1182 non-null   int64  
 2   Injection_Time            1182 non-null   float64
 3   Filling_Time              1182 non-null   float64
 4   Plasticizing_Time         1182 non-null   float64
 5   Cycle_Time                1182 non-null   float64
 6   Clamp_Close_Time          1182 non-null   float64
 7   Cushion_Position          1182 non-null   float64
 8   Plasticizing_Position     1182 non-null   float64
 9   Clamp_Open_Position       1182 non-null   float64
 10  Max_Injection_Speed       1182 non-null   float64
 11  Max_Screw_RPM             1182 non-null   float64
 12  Average_Screw_RPM         1182 non-null   float64
 13  Max_Injection_Pressure    1182 non-null   float64
 14  Max_Switch_Over_Pre

In [28]:
# 데이터 타입 확인
import numpy as np

num_rg3 = train_rg3.select_dtypes(include = np.number)
num_rg3.columns

Index(['Unnamed: 0', 'PassOrFail', 'Injection_Time', 'Filling_Time',
       'Plasticizing_Time', 'Cycle_Time', 'Clamp_Close_Time',
       'Cushion_Position', 'Plasticizing_Position', 'Clamp_Open_Position',
       'Max_Injection_Speed', 'Max_Screw_RPM', 'Average_Screw_RPM',
       'Max_Injection_Pressure', 'Max_Switch_Over_Pressure',
       'Max_Back_Pressure', 'Average_Back_Pressure', 'Barrel_Temperature_1',
       'Barrel_Temperature_2', 'Barrel_Temperature_3', 'Barrel_Temperature_4',
       'Barrel_Temperature_5', 'Barrel_Temperature_6', 'Hopper_Temperature',
       'Mold_Temperature_3', 'Mold_Temperature_4'],
      dtype='str')

In [12]:
# 결측치 확인

train_rg3.isnull().sum() / len(train_cn7)

Unnamed: 0                  0.0
PassOrFail                  0.0
Injection_Time              0.0
Filling_Time                0.0
Plasticizing_Time           0.0
Cycle_Time                  0.0
Clamp_Close_Time            0.0
Cushion_Position            0.0
Plasticizing_Position       0.0
Clamp_Open_Position         0.0
Max_Injection_Speed         0.0
Max_Screw_RPM               0.0
Average_Screw_RPM           0.0
Max_Injection_Pressure      0.0
Max_Switch_Over_Pressure    0.0
Max_Back_Pressure           0.0
Average_Back_Pressure       0.0
Barrel_Temperature_1        0.0
Barrel_Temperature_2        0.0
Barrel_Temperature_3        0.0
Barrel_Temperature_4        0.0
Barrel_Temperature_5        0.0
Barrel_Temperature_6        0.0
Hopper_Temperature          0.0
Mold_Temperature_3          0.0
Mold_Temperature_4          0.0
dtype: float64

In [35]:
# Target 제외 중복 -> 동일한 공정조건으로 여러 제품이 생산됐는지 확인

train_rg3.drop(columns="PassOrFail").duplicated().sum()

np.int64(0)

In [36]:
# 동일 X + 동일 Target -> 반복 생산 데이터일 가능성
train_rg3.duplicated().sum()

np.int64(0)

In [37]:
# 동일 X + 다른 Target -> 같은 공정조건인데 품질 결과가 달라지는 노이즈 또는 미관측 변수 가능성

train_rg3.groupby(
    list(train_rg3.columns.drop("PassOrFail"))
)["PassOrFail"].nunique().gt(1).sum()

np.int64(0)

In [20]:
# target 분포 확인

train_rg3['PassOrFail'].value_counts()

PassOrFail
0    1157
1      25
Name: count, dtype: int64

In [38]:
# target 분포 비율 확인
train_cn7["PassOrFail"].value_counts(normalize=True)

PassOrFail
0    0.985962
1    0.014038
Name: proportion, dtype: float64

In [41]:
# 시계열 데이터
# 시계열 컬럼 존재 여부 확인
[col for col in train_rg3.columns 
 if any(x in col.lower() for x in ["date", "time", "timestamp"])]

['Injection_Time',
 'Filling_Time',
 'Plasticizing_Time',
 'Cycle_Time',
 'Clamp_Close_Time']

In [ ]:
# 날짜/시각 컬럼 확인 -> 없음을 확인
[col for col in train_rg3.columns 
 if any(x in col.lower() for x in ["date", "datetime", "timestamp"])]

[]

In [44]:
# 설비 구조
# 설비 식별자 컬럼 확인
[col for col in train_rg3.columns
 if any(x in col.lower() for x in ["machine_id", "equipment_id", "device_id", "line_id"])]

[]

In [46]:
[col for col in train_rg3.columns
 if any(x in col.lower() for x in ["lot", "batch", "group", "product_id"])]

[]